# Kindle Review Sentiment Analysis

In [198]:
import kagglehub
path = kagglehub.dataset_download("meetnagadia/amazon-kindle-book-review-for-sentiment-analysis")

Using Colab cache for faster access to the 'amazon-kindle-book-review-for-sentiment-analysis' dataset.


In [199]:
import os
os.listdir(path)

['all_kindle_review .csv', 'preprocessed_kindle_review .csv']

In [200]:
import pandas as pd
df=pd.read_csv(path+'/all_kindle_review .csv')

In [201]:
df.head(2)

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400


In [202]:
df.shape

(12000, 11)

In [203]:
df=df[['reviewText','rating']]

In [204]:
df.sample(1)

,reviewText,rating
9213,Although one of the main characters was emotio...,3


In [205]:
df.isnull().sum()

,0
reviewText,0
rating,0


In [206]:
df.duplicated().sum()

np.int64(0)

## There are no nulls and duplicates found in dataset

In [207]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   reviewText  12000 non-null  object
 1   rating      12000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 187.6+ KB


In [208]:
df['rating']=df['rating'].astype('int16')

In [209]:
df['reviewText'][0]

'Jace Rankin may be short, but he\'s nothing to mess with, as the man who was just hauled out of the saloon by the undertaker knows now. He\'s a famous bounty hunter in Oregon in the 1890s who, when he shot the man in the saloon, just finished a years long quest to avenge his sister\'s murder and is now trying to figure out what to do next. When the snotty-nosed farm boy he just rescued from a gang of bullies offers him money to kill a man who forced him off his ranch, he reluctantly agrees to bring the man to justice, but not to kill him outright. But, first he needs to tell his sister\'s widower the news.Kyla "Kyle" Springer Bailey has been riding the trails and sleeping on the ground for the past month while trying to find Jace. She wants revenge on the man who killed her husband and took her ranch, amongst other crimes, and she\'s not so keen on the detour Jace wants to take. But she realizes she\'s out of options, so she hides behind her boy persona as best she can and tries to ke

In [210]:
import re
def remove_impurity(col):
  col=col.lower()
  col=re.sub('<.*?>',' ',col)
  col=re.sub('http?://\S+|www\.\S+',' ',col)
  return re.sub('[^a-zA-Z]',' ',col)
df['reviewText']=df['reviewText'].apply(remove_impurity)

<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:5: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_2222/3561170560.py:5: SyntaxWarning: invalid escape sequence '\S'
  col=re.sub('http?://\S+|www\.\S+',' ',col)


In [211]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
words=stopwords.words('english')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [212]:
nltk.download('punkt_tab')
def remove_stopwords(col):
  L=[]
  tokens=word_tokenize(col)
  for i in tokens:
    if i not in words:
      L.append(i)
  return " ".join(L)
df['reviewText']=df['reviewText'].apply(remove_stopwords)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [213]:
from nltk.stem import SnowballStemmer
sbs=SnowballStemmer('english')
def stemmer(col):
  L=[]
  tokens=word_tokenize(col)
  for i in tokens:
    L.append(sbs.stem(i))
  return " ".join(L)
df['reviewText']=df['reviewText'].apply(stemmer)

In [214]:
df['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int16)

In [215]:
df['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


In [216]:
def sentiment(value):
  if value in [1,2]:
    return 0
  else:
    return 1
df['rating']=df['rating'].apply(sentiment)

In [217]:
df['rating'].value_counts()

,count
rating,
1,8000
0,4000


## 0 -> Negative
## 1 -> Positive

In [218]:
df.tail(2)

,reviewText,rating
11998,tri use charg kindl even regist charg complet ...,0
11999,take instruct look often hidden world sexual b...,1


In [219]:
x=df[['reviewText']]
y=df['rating']

In [220]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [221]:
x_train=pd.DataFrame(x_train)
x_test=pd.DataFrame(x_test)

In [222]:
x_train.head()

,reviewText
9182,look forward book came doubl space everi parag...
11091,alreadi own book spous forgot alreadi part lib...
6428,cool forgot request rate came make mine unreli...
288,short short stori basic scene parti one night ...
2626,secret servic agent secrest even longer servic...


In [223]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=2500,ngram_range=(1,2))
x_train=tf.fit_transform(x_train['reviewText'])
x_test=tf.transform(x_test['reviewText'])

In [235]:
import dill
with open('tf.pkl','wb') as f:
  dill.dump(tf,f)

In [224]:
x_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 385208 stored elements and shape (9600, 2500)>

In [225]:
from sklearn.linear_model import LogisticRegression
model1=LogisticRegression()
model1.fit(x_train,y_train)

LogisticRegression()

In [226]:
train_pred=model1.predict(x_train)
test_pred=model1.predict(x_test)

In [227]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
print(accuracy_score(y_train,train_pred))
print(accuracy_score(y_test,test_pred))

0.8770833333333333
0.84


In [228]:
print('For Training : \n',confusion_matrix(y_train,train_pred))
print('For Testing : \n',confusion_matrix(y_test,test_pred))

For Training : 
 [[2346  851]
 [ 329 6074]]
For Testing : 
 [[ 522  281]
 [ 103 1494]]


In [229]:
print('Training Report :',classification_report(y_train,train_pred))
print('Testing Report :',classification_report(y_test,test_pred))

Training Report :               precision    recall  f1-score   support

           0       0.88      0.73      0.80      3197
           1       0.88      0.95      0.91      6403

    accuracy                           0.88      9600
   macro avg       0.88      0.84      0.86      9600
weighted avg       0.88      0.88      0.87      9600

Testing Report :               precision    recall  f1-score   support

           0       0.84      0.65      0.73       803
           1       0.84      0.94      0.89      1597

    accuracy                           0.84      2400
   macro avg       0.84      0.79      0.81      2400
weighted avg       0.84      0.84      0.83      2400



In [236]:
with open('model.pkl','wb') as f:
  dill.dump(model1,f)